In [2]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain.schema import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import json
from langchain import hub
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.llms import Ollama

In [3]:
# 預處理documents
json_file_path = "example_documents.json"
with open(json_file_path, 'r', encoding='utf-8') as file:
    data = json.load(file)

In [4]:
CHUNK_SIZE = 100
CHUNK_OVERLAP = 10
All_passage_id = {}
def process_documents(data, num_of_docs):
    documents = []
    original_content_dict = {}
    # 先處理每個文檔
    for doc in data["documents"]:
        original_content = doc["content"]
        
        # 創建基本的Document對象
        document = Document(
            page_content=original_content,
            metadata={
                "id": doc["id"],
                "title": doc["title"],
                "author": doc["author"],
                #"original_content": original_content  # 保存原始內容
            }
        )
        documents.append(document)
        original_content_dict[doc["id"]] = original_content
    
    # 創建text splitter並添加索引追踪
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE, 
        chunk_overlap=CHUNK_OVERLAP,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    # 分割文檔並添加索引信息
    splits = text_splitter.split_documents(documents)

    # 為每個分割添加索引信息
    # count_for_each_doc = {{}}
    num_of_passages = 0
    for doc in splits:

        chunk_content = doc.page_content
        start_idx = original_content_dict[doc.metadata["id"]].find(chunk_content)
        end_idx = start_idx + len(chunk_content) - 1

        # if(doc.metadata["id"] not in count_for_each_doc):
        #     count_for_each_doc[doc.metadata["id"]] = 0
        
        # Stride=len(doc.page_content)
        # start_idx = count_for_each_doc[doc.metadata["id"]]

        # end_idx = start_idx + Stride -1
        # count_for_each_doc[doc.metadata["id"]] = end_idx + 1
        # 更新metadata，添加索引信息
        doc.metadata.update({
            "chunk_start_idx": start_idx,
            "chunk_end_idx": end_idx,
            "passage_id": num_of_passages
        })
        All_passage_id[num_of_passages] = {
            "start_idx": start_idx,
            "end_idx": end_idx,
            "source_doc_id": doc.metadata["id"],
        }
        num_of_passages += 1
        # 可以選擇刪除原始內容以節省空間
    
    return splits

# 有幾個文檔，就有幾個count_for_each_doc

all_splits = process_documents(data, len(data["documents"]))

# 打印處理後的文件，包含索引信息
for split in all_splits:
    print(f"\n文檔ID: {split.metadata['id']}")
    print(f"標題: {split.metadata['title']}")
    print(f"內容: {split.page_content}")
    print(f"索引範圍: {split.metadata['chunk_start_idx']} 到 {split.metadata['chunk_end_idx']}")


文檔ID: doc1
標題: 2024美國總統大選候選人分析
內容: 2024年美國總統大選的主要候選人包括現任總統喬·拜登和前總統唐納德·川普。拜登在競選過程中強調其首個任期的政績，包括推動基礎建設法案、通過通膨削減法案，以及在國際舞台上重建美國領導地位。他的競選主軸
索引範圍: 0 到 99

文檔ID: doc1
標題: 2024美國總統大選候選人分析
內容: 導地位。他的競選主軸著重於保護民主、經濟復甦、氣候行動和社會公平。副總統賀錦麗在競選中扮演重要角色，特別是在爭取年輕選民和少數族裔選民方面。另一方面，川普的競選策略則聚焦於批評拜登政府的邊境政策、通貨
索引範圍: 90 到 189

文檔ID: doc1
標題: 2024美國總統大選候選人分析
內容: 政府的邊境政策、通貨膨脹問題，以及能源政策。他承諾若重返白宮將採取更強硬的移民政策，並重新評估美國的國際承諾，特別是在氣候變遷協議方面。此外，川普也持續質疑2020年大選的公正性，這個議題在其支持者中
索引範圍: 180 到 279

文檔ID: doc1
標題: 2024美國總統大選候選人分析
內容: 這個議題在其支持者中引起強烈共鳴。兩位候選人的民調支持度起伏不定，反映出美國選民對國家未來方向的分歧。
索引範圍: 270 到 320

文檔ID: doc2
標題: 2024大選關鍵議題探討
內容: 2024年美國總統大選中，經濟議題和通貨膨脹成為選民最關心的重點。拜登陣營強調在其執政期間創造了創紀錄的就業機會，失業率維持在低點，並通過多項經濟振興方案。然而，持續的高通膨和生活成本上升讓許多選民感
索引範圍: 0 到 99

文檔ID: doc2
標題: 2024大選關鍵議題探討
內容: 成本上升讓許多選民感到不安。川普陣營則抨擊拜登的經濟政策導致通膨失控，並提出降稅和減少政府支出的主張。移民政策是另一個激烈辯論的焦點，在南部邊境危機持續之際，兩位候選人提出截然不同的解決方案。拜登支持
索引範圍: 90 到 189

文檔ID: doc2
標題: 2024大選關鍵議題探討
內容: 的解決方案。拜登支持全面性的移民改革，包括為無證移民提供公民身份途徑，同時加強邊境管理。川普則主張建立更嚴格的邊境管制，並承諾完成邊境牆的建設。在外交政策方面，美中關係、俄烏戰爭、以及台海局勢都是關鍵
索引範圍: 180 到

In [5]:

embeddings = OllamaEmbeddings(model="nomic-embed-text")
vector_store = FAISS.from_documents(all_splits, embeddings)

C:\Users\kevin\AppData\Local\Temp\ipykernel_22416\4178560103.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="nomic-embed-text")


In [6]:
# 建立QA系統
# See full prompt at https://smith.langchain.com/hub/langchain-ai/retrieval-qa-chat
llm = Ollama(model="llama3.1:8b")
retrieval_qa_chat_prompt = hub.pull("langchain-ai/retrieval-qa-chat")

combine_docs_chain = create_stuff_documents_chain(llm, retrieval_qa_chat_prompt)
rag_chain = create_retrieval_chain(vector_store.as_retriever(), combine_docs_chain)
# predined question
question = ["這個議題有哪些重要人物?", "這個事件的時間軸是?", "這個事件的結果是?", "這個事件的影響是?", "這個事件的意義是?"]
final_result = []
for q in question:
    result = rag_chain.invoke({"input": q})
    final_result.append(result)


C:\Users\kevin\AppData\Local\Temp\ipykernel_22416\4180778331.py:3: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  llm = Ollama(model="llama3.1:8b")
d:\Commonground_reference_system\.venv\lib\site-packages\langsmith\client.py:221: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [7]:
print(final_result)


[{'input': '這個議題有哪些重要人物?', 'context': [Document(metadata={'id': 'doc1', 'title': '2024美國總統大選候選人分析', 'author': '王政治分析師', 'chunk_start_idx': 270, 'chunk_end_idx': 320, 'passage_id': 3}, page_content='這個議題在其支持者中引起強烈共鳴。兩位候選人的民調支持度起伏不定，反映出美國選民對國家未來方向的分歧。'), Document(metadata={'id': 'doc2', 'title': '2024大選關鍵議題探討', 'author': '李政策研究員', 'chunk_start_idx': 360, 'chunk_end_idx': 404, 'passage_id': 8}, page_content='學並支持化石燃料產業。民主制度的健全性也成為選戰焦點，特別是在選舉安全和投票權方面的爭議。'), Document(metadata={'id': 'doc1', 'title': '2024美國總統大選候選人分析', 'author': '王政治分析師', 'chunk_start_idx': 0, 'chunk_end_idx': 99, 'passage_id': 0}, page_content='2024年美國總統大選的主要候選人包括現任總統喬·拜登和前總統唐納德·川普。拜登在競選過程中強調其首個任期的政績，包括推動基礎建設法案、通過通膨削減法案，以及在國際舞台上重建美國領導地位。他的競選主軸'), Document(metadata={'id': 'doc2', 'title': '2024大選關鍵議題探討', 'author': '李政策研究員', 'chunk_start_idx': 90, 'chunk_end_idx': 189, 'passage_id': 5}, page_content='成本上升讓許多選民感到不安。川普陣營則抨擊拜登的經濟政策導致通膨失控，並提出降稅和減少政府支出的主張。移民政策是另一個激烈辯論的焦點，在南部邊境危機持續之際，兩位候選人提出截然不同的解決方案。拜登支持')], 'answer': '喬·拜登（Joe Bi

In [8]:
def format_qa_result(final_result): 
    model_input_json = {}
    for i,qa in enumerate(final_result,1):
        question_key = f"Question_{i}"
        model_input_json[question_key] = {}
        model_input_json[question_key]["Question"] = qa['input']
        model_input_json[question_key]["Answer"] = qa['answer']
        model_input_json[question_key]["cited_passages"] = {}
        for j,context in enumerate(qa['context'],1):
            model_input_json[question_key]["cited_passages"][f"passage{j}"] = {
                "start_idx": context.metadata['chunk_start_idx'],
                "end_idx": context.metadata['chunk_end_idx'],
                "source_doc_id": context.metadata['id'],
                "passage_id": context.metadata['passage_id']
            }
    

    return model_input_json
model_input_json = format_qa_result(final_result)

In [9]:
print(model_input_json)

{'Question_1': {'Question': '這個議題有哪些重要人物?', 'Answer': '喬·拜登（Joe Biden）、唐納德·川普（Donald Trump）、學者不明（可能是指支持化石燃料產業的政客或政策制定者）', 'cited_passages': {'passage1': {'start_idx': 270, 'end_idx': 320, 'source_doc_id': 'doc1', 'passage_id': 3}, 'passage2': {'start_idx': 360, 'end_idx': 404, 'source_doc_id': 'doc2', 'passage_id': 8}, 'passage3': {'start_idx': 0, 'end_idx': 99, 'source_doc_id': 'doc1', 'passage_id': 0}, 'passage4': {'start_idx': 90, 'end_idx': 189, 'source_doc_id': 'doc2', 'passage_id': 5}}}, 'Question_2': {'Question': '這個事件的時間軸是?', 'Answer': '根據上下文，我們可以推測這次選舉是在美國進行的，並且正在進入投票階段。可能是2024年美國總統大選，或者是其他州份的大選，但我無法確定具體時間點。', 'cited_passages': {'passage1': {'start_idx': 90, 'end_idx': 189, 'source_doc_id': 'doc3', 'passage_id': 10}, 'passage2': {'start_idx': 180, 'end_idx': 279, 'source_doc_id': 'doc3', 'passage_id': 11}, 'passage3': {'start_idx': 270, 'end_idx': 348, 'source_doc_id': 'doc3', 'passage_id': 12}, 'passage4': {'start_idx': 90, 'end_idx': 189, 'source_doc_id': 'doc1', 'passage_id

In [17]:
#把QA問答集變成非結構化問答格式 

model_input_string = ""
for key,value in model_input_json.items():
    model_input_string += f"{key[-1]}. {value['Question']}\n"
    model_input_string += f"答: {value['Answer']}"
    if(model_input_string[-1] == "。"):
        model_input_string = model_input_string[:-1]
    for context in value['cited_passages']:
        #print(context)
        model_input_string += f"[{value['cited_passages'][context]['passage_id']}]"
    model_input_string += "\n"
print(model_input_string)


1. 這個議題有哪些重要人物?
答: 喬·拜登（Joe Biden）、唐納德·川普（Donald Trump）、學者不明（可能是指支持化石燃料產業的政客或政策制定者）[3][8][0][5]
2. 這個事件的時間軸是?
答: 根據上下文，我們可以推測這次選舉是在美國進行的，並且正在進入投票階段。可能是2024年美國總統大選，或者是其他州份的大選，但我無法確定具體時間點[10][11][12][1]
3. 這個事件的結果是?
答: 此次大選結果將決定美國未來四年的政策方向，也將深刻影響全球地緣政治格局[10][11][12][3]
4. 這個事件的影響是?
答: 此次大選結果不僅將決定美國未來四年的政策方向，也將深刻影響全球地緣政治格局[10][11][12][3]
5. 這個事件的意義是?
答: 這個大選結果不僅決定美國未來四年的政策方向，也將深刻影響全球地緣政治格局[11][12][10][8]



In [8]:
from langchain_core.prompts import PromptTemplate

one_shot_prompt = PromptTemplate.from_template(
    """
    有一系列的QA問題 每一個問題都有他對應的問題和答案，並且也有他這個問題所對應到的答案是根據哪個文檔的哪個段落來的，我希望你透過這些資訊，把這個QA問題及形成一份完整的內容，我的這個QA問題集會問出這個事件的脈絡和經過
    ，此外對於你寫出來的摘要對應到的每一段文字都要說明這段文字是來自哪個文檔的哪個段落

    以下一個列出QA問題和答案的範例(以美國大選為例)
    1. 這個事件中的主要人物有誰?
    答: 這個事件中有川普、賀錦麗、拜登、哈里斯(根據[doc_id: 1, chunk_start_idx: 10, chunk_end_idx: 20])
    2. 這個事件的時間軸是?
    答: 這個事件的時間軸是2024年11月5日(根據[doc_id: 2, chunk_start_idx: 30, chunk_end_idx: 40])
    3. 這個事件的經過是?
    答: 這個事件的經過是...(根據[doc_id: 3, chunk_start_idx: 50, chunk_end_idx: 60])
    4. 這個事件的結果是?
    答: 這個事件的結果是...(根據[doc_id: 4, chunk_start_idx: 70, chunk_end_idx: 80])
    5. 這個事件的影響是?
    答: 這個事件的影響是...(根據[doc_id: 5, chunk_start_idx: 90, chunk_end_idx: 100])
    6. 這個事件的意義是?
    答: 這個事件的意義是...(根據[doc_id: 6, chunk_start_idx: 30, chunk_end_idx: 120])

    摘要範例
    在2024年的美國大選中，川普、賀錦麗、拜登與哈里斯成為這場政治較量中的核心人物（根據[doc_id: 1, chunk_start_idx: 10, chunk_end_idx: 20]）。這次選舉於2024年11月5日正式展開，這一天標誌了美國政治舞台上的重要轉折點，無數選民的選擇集中在這一刻，訴說著美國選舉體制的意涵（根據[doc_id: 2, chunk_start_idx: 30, chunk_end_idx: 40]）。
    在競選過程中，川普與拜登多次激烈辯論，展示了彼此的政策願景及施政方針。川普以一貫強硬的立場面對公共政策問題，而拜登則強調社會團結和環保議題。另一方面，賀錦麗與哈里斯分別代表兩黨輔佐競選，致力於拉攏支持者，讓民眾能夠更加明確地理解兩黨的政策異同（根據[doc_id: 3, chunk_start_idx: 50, chunk_end_idx: 60]）。在這樣的激烈交鋒中，美國選民的支持逐漸傾向於拜登，最終，他在選舉中勝出，成功成為美國總統，而川普未能再度連任。這一結果引起了各地選民的激烈反應，許多地方爆發了不同政見的抗議活動，表現出民眾對此次選舉結果的複雜情緒（根據[doc_id: 4, chunk_start_idx: 70, chunk_end_idx: 80]）。
    隨著拜登的當選，美國的內政外交迎來了新氣象，他在上任後推行了一系列環保和經濟改革，逐步改變了美國的國內外政策格局，這些舉措也帶來了對未來美國發展方向的深刻影響（根據[doc_id: 5, chunk_start_idx: 90, chunk_end_idx: 100]）。而這次大選的意義並不僅限於政權更替，更是一場對於美國民主體制的深刻檢視。它顯示了美國選民在當代政治中愈發關鍵的角色，並且凸顯了兩黨對立的現實，激起了關於政治分歧與社會團結的討論，這些討論將在未來繼續發酵，對美國的民主制度和社會發展產生深遠的影響（根據[doc_id: 6, chunk_start_idx: 30, chunk_end_idx: 120]）。

    現在請你根據這個範例，把我的QA問題和答案轉換成一份完整的內容

    我的QA問題和答案
    {input}



    """
)

answer_to_abstract_chain = one_shot_prompt | llm
result = answer_to_abstract_chain.invoke({"input": model_input_json})
print(result)

我將根據你的QA問題和答案，產生一份完整的內容。

嬰兒發展研究是一個重要的學科領域，它對於我們了解嬰兒成長、發育與認知能力的形成有著深遠的影響。根據上下文，我們可以得知：嬰兒發展研究的一個重要發現是，互動的質量比數量更為重要，對神經網絡的形成有決定性影響。

早期人際互動對嬰兒後續認知、情感和行為發展具有決定性影響。這一點可以從文中最後一句話「指出，互動的質量比數多」可推斷出，嬰兒發展研究可能發現的是：早期人際互動對嬰兒後續認知、情感和行為發展具有決定性影響。

對神經網絡的形成有決定性的影響。這是嬰兒發展研究的一個重要發現，它表明了早期人際互動對嬰兒後續認知、情感和行為發展的影響。

綜上所述，我們可以得知：嬰兒發展研究有一系列重要的發現，包括互動的質量比數量更為重要、對神經網絡的形成有決定性影響，以及早期人際互動對嬰兒後續認知、情感和行為發展具有決定性影響。

對應到的每一段文字都是來自[doc_id: doc3, chunk_start_idx: 49, chunk_end_idx: 58]


In [9]:
prompt = PromptTemplate.from_template(
    """
你是一位專業的文章撰寫者，請你根據以下的問答內容，撰寫一份完整且連貫的摘要報告。

問答內容：
{input}

{format_instructions}
撰寫要求：
1. 內容結構：
   - 按照時間順序或邏輯順序組織內容
   - 確保段落之間的轉折自然
   - 使用適當的連接詞來增加文章流暢度

2. 引用格式：
   - 每個論述都必須標註來源：（根據[1][2]） 其中的[1]代表他引用了passage_number=1的passage，[2]代表他引用了passage_number=2的passage
   - 來源標註要放在相關敘述的結尾處
   - 如果一個段落包含多個來源的內容，需分別標註

3. 寫作風格：
   - 使用客觀、專業的語氣
   - 避免重複QA中的問題形式
   - 將問答轉化為敘事性的描述

4. 內容完整性：
   - 確保涵蓋所有QA中的重要信息
   - 適當整合相關信息，避免過度分散
   - 在不同觀點間取得平衡

請根據以上要求，將QA內容改寫成一份連貫的摘要報告。報告應該能讓讀者清楚理解事件的來龍去脈，同時保持專業性和可信度。

輸出格式示例：
[完整的敘事性摘要，每個論述都需包含來源標註]

注意：請確保每個重要信息都有對應的文檔來源標註，且內容的組織要合乎邏輯，讓讀者能夠輕鬆理解整個事件的發展脈絡。
   

"""
   input_variables = ["input"],
   partial_variables={"format_instructions": format_instructions}
)

answer_to_abstract_chain = prompt | llm
result = answer_to_abstract_chain.invoke({"input": final_result_qa_string, "format_instructions": format_instructions})

print(result)

嬰兒發展研究在近年來取得了令人關注的進展。在這些研究中，尤其提到了互動的質量對神經網絡形成有決定性的影響。根據[doc_id: doc3, chunk_start_idx: 49, chunk_end_idx: 58]的文獻記錄，這一發現揭示了早期人際互動對嬰兒後續認知、情感和行為發展具有深遠的影響。

更進一步地，研究者還將注意力聚焦於嬰兒與其環境之間的相互作用。這些相互作用不僅僅是量上的問題，而是質量上的關鍵。根據[doc_id: doc3, chunk_start_idx: 49, chunk_end_idx: 58]的文獻記錄，高質量的互動能夠提供嬰兒更豐富的經驗和知識積累，並對其神經網絡的發展產生持久性的影響。

對於這一領域的研究來說，這些發現不僅深刻而且重要。它們向我們證明了早期互動對嬰兒成長的決定性作用，同時也提到了高質量的互動是如何實踐到嬰兒發展中的關鍵因素。

因此，嬰兒發展研究在這些年來取得了顯著的進展。它們不僅提供了對早期互動對嬰兒發展的深刻洞察，也提出了對高質量互動的重視。根據[doc_id: doc3, chunk_start_idx: 49, chunk_end_idx: 58]的文獻記錄，這些發現對於我們理解嬰兒成長和發展具有持續性的影響，而這將是我們未來研究的重要方向。


In [18]:
def load_few_shot(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return file.read()
few_shot_example = load_few_shot("few-shot-example.txt")


In [19]:
print(model_input_string)


1. 這個議題有哪些重要人物?
答: 喬·拜登（Joe Biden）、唐納德·川普（Donald Trump）、學者不明（可能是指支持化石燃料產業的政客或政策制定者）[3][8][0][5]
2. 這個事件的時間軸是?
答: 根據上下文，我們可以推測這次選舉是在美國進行的，並且正在進入投票階段。可能是2024年美國總統大選，或者是其他州份的大選，但我無法確定具體時間點[10][11][12][1]
3. 這個事件的結果是?
答: 此次大選結果將決定美國未來四年的政策方向，也將深刻影響全球地緣政治格局[10][11][12][3]
4. 這個事件的影響是?
答: 此次大選結果不僅將決定美國未來四年的政策方向，也將深刻影響全球地緣政治格局[10][11][12][3]
5. 這個事件的意義是?
答: 這個大選結果不僅決定美國未來四年的政策方向，也將深刻影響全球地緣政治格局[11][12][10][8]



In [47]:
from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate.from_template(
    '''
    您是一位專業的文章撰寫者，請根據以下的問答內容，撰寫一份完整且連貫的摘要報告。

以下是一個問答集包含一系列的問題和答案：
{input}

撰寫要求：
1. 內容結構：
   - 按照時間順序或邏輯順序組織內容
   - 確保段落之間的轉折自然
   - 使用適當的連接詞來增加文章流暢度
   - 除了摘要內容之外不要有任何多餘的文字
   - 在輸出的內容中不要有任何除了摘要內容和引用標註之外的描述性文字
   - 不可以用列點的方式來輸出

2. 引用格式：
   - 每個論述都必須標註來源：[n]
   - 如果一個段落包含多個來源的內容，需分別標註

3. 寫作風格：
   - 使用客觀、專業的語氣
   - 避免重複QA中的問題形式
   - 將問答轉化為敘事性的描述

4. 內容完整性：
   - 確保涵蓋所有QA中的重要信息
   - 適當整合相關信息，避免過度分散
   - 在不同觀點間取得平衡

以下是範例問答輸入和範例摘要示例輸出：
{examples}

請根據上述範例格式和要求，輸出帶有引用的摘要。
'''
)

answer_to_abstract_chain = prompt | llm
result = answer_to_abstract_chain.invoke({"input": model_input_string, "examples": few_shot_example})
print(result)




2008年金融危機中，小布希總統、財政部長保爾森、聯準會主席柏南克和雷曼兄弟CEO理查德·富爾德成為關鍵人物[1]。危機於2008年9月15日達到頂點，當天雷曼兄弟宣布破產[2]，隨後引發連鎖反應，造成全球金融市場劇烈動盪[4]。美國政府被迫採取緊急措施，推出規模達7000億美元的紓困方案，試圖穩定金融市場[6]。然而，全球經濟仍陷入嚴重衰退，美國失業率攀升至10%以上，多個金融機構倒閉或被收購[8]。這場危機促使各國政府加強金融監管，建立更嚴格的風險控制機制，徹底改變了全球金融監管格局[10]。其深遠意義在於促使人們重新思考金融體系的脆弱性，推動了全球金融改革，並強化了國際金融合作機制[11]。

相比之下，這場全球大流行疫情發生在2019年12月，在中國武漢首次被發現[2]。到2020年3月11日，被世衛組織正式宣布為全球大流行[3]，隨後迅速蔓延至全球各地。各國領導人如習近平、川普等扮演關鍵角色，採取了嚴格的防疫措施，但仍然造成全球過億人確診和重創[5][7]。這場危機改變了人們的生活方式，加速了遠距工作等數位轉型趨勢，促進了醫療科技的發展，並凸顯了全球衛生治理的重要性和國際合作模式的變化[9][10]。

兩起事件都表明了公共健康、經濟、政治等各個層面對危機應對能力的挑戰。


In [24]:
from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate.from_template(
    '''
    您是一位專業的文章撰寫者，請根據實際問答集內容，撰寫一份完整且連貫的摘要報告。

### 範例格式說明 ###
以下是兩個範例，展示如何將問答轉換為摘要：
{examples}

### 實際任務 ###
現在，請根據以下實際問答集內容的問答內容，按照上述範例的格式撰寫摘要：
{input}

注意：
1. 內容結構：
   - 按照時間順序或邏輯順序組織內容
   - 確保段落之間的轉折自然
   - 使用適當的連接詞來增加文章流暢度
   

2. 引用格式：
   - 每個論述都必須標註來源：[n]
   - 如果一個段落包含多個來源的內容，需分別標註

3. 寫作風格：
   - 使用客觀、專業的語氣
   - 避免重複QA中的問題形式
   - 將問答轉化為敘事性的描述

4. 內容完整性：
   - 確保涵蓋所有QA中的重要信息
   - 適當整合相關信息，避免過度分散
   - 在不同觀點間取得平衡

5. 嚴格遵守以下規則限制
   - 除了摘要內容之外不要有任何多餘的文字
   - 在輸出的內容中不要有任何除了摘要內容和引用標註之外的描述性文字
   - 不可以用列點的方式來輸出，
   - 不要輸出「以下是摘要:」或者是「摘要報告」之類的文字，請直接輸出摘要
'''
)

answer_to_abstract_chain = prompt | llm
result = answer_to_abstract_chain.invoke({"input": model_input_string, "examples": few_shot_example})
print(result)




根據你的指示，我將撰寫一個完整且連貫的摘要報告。

**摘要**

2024年美國總統大選中的主要人物包括喬·拜登和唐納德·川普。這場選舉於2024年的某個時間點進行，並將對美國未來四年的政策方向產生深遠影響。選舉結果不僅決定美國未來的領導人，也將對全球地緣政治格局造成重要變動。此外，這次選舉也將加速美國內部政策方向的轉折，例如能源政策和經濟發展等方面。

**摘要報告**

喬·拜登和唐納德·川普在2024年的美國總統大選中扮演了關鍵角色[3][8][0][5]。此次選舉將對美國未來四年的政策方向產生深遠影響，並將對全球地緣政治格局造成重要變動[10][11][12][3]。

選舉結果不僅決定美國未來的領導人，也將對全球地緣政治格局造成重要變動。此外，這次選舉也將加速美國內部政策方向的轉折，例如能源政策和經濟發展等方面[10][11][12][3]。

選舉過程中的影響將是非常重要的，並且將對美國未來的發展方向產生重大影響。最終的結果將決定美國未來四年的政策方向，並將對全球地緣政治格局造成深遠影響。

**結論**

2024年美國總統大選中，喬·拜登和唐納德·川普將在關鍵角色中扮演重要作用。這次選舉結果將決定美國未來四年的政策方向，並將對全球地緣政治格局造成深遠影響。此外，這次選舉也將加速美國內部政策方向的轉折，例如能源政策和經濟發展等方面。

**參考資料**

[3][8][0][5]、[10][11][12][3]、[11][12][10][8]


In [48]:
from langchain.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Dict, Optional
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
class Passage(BaseModel):
    start_idx: int
    end_idx: int
    source: str

class CitedPassages(BaseModel):
    passages: Dict[str, Passage] = Field(description="引用的段落信息")

class SummaryContent(BaseModel):
    Q_and_A_dataset: str = Field(description="文章的QA問答")
    summary_and_citation_output: str = Field(description="摘要內容，包含引用標記[1][2]等")
    cited_passages: CitedPassages

class OutputFormat(BaseModel):
    Summary: SummaryContent

parser = PydanticOutputParser(pydantic_object=OutputFormat)

prompt = PromptTemplate(
    input_variables = ["input"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
    template="""你是一位專業的文章撰寫者，請你根據以下的問答內容，撰寫一份完整且連貫的摘要報告。

問答內容：
{input}

{format_instructions}
'''

請嚴格按照以下JSON格式輸出，不要有任何多餘的文字：
{{
    "Summary": {{
        "Q_and_A_dataset": "原始QA数据集内容",
        "summary_and_citation_output": "摘要内容，使用[1][2]等標記引用passage_id",
        "cited_passages": {{
            "passage1": {{
                "start_idx": 0,
                "end_idx": 10,
                "source_doc_id": "文件來源"
            }},
            "passage2": {{
                "start_idx": 11,
                "end_idx": 20,
                "source_doc_id": "文件來源"
            }}
        }}
    }}
}}
'''

"""
    
)
summary_and_citation_chain = prompt | llm | parser
result = summary_and_citation_chain.invoke({"input": model_input_json, "format_instructions": parser.get_format_instructions()})
print(result)




NameError: name 'BaseModel' is not defined